In [1]:
#!/usr/bin/env python3
"""
March Machine Learning Mania 2026 - 4モデル + OppAdj + Momentum
================================================================
opp_adj版 (LB=0.08525) ベース。

追加: 直近10試合のモメンタム特徴量
  - シーズン終盤の調子（勝率、得失点差、FG%、効率）
  - ケガ・調子の波を捕捉
"""

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
import warnings
import os
import sys

warnings.filterwarnings('ignore')

# ============================================================
# 0. INSTALL goto_conversion + XGBoost + LightGBM
# ============================================================
try:
    import goto_conversion
except ImportError:
    import subprocess
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'goto-conversion', '-q'])
    except subprocess.CalledProcessError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'goto-conversion', '-q', '--break-system-packages'])
    import goto_conversion

HAS_XGB = False
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    try:
        import subprocess
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])
        except subprocess.CalledProcessError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q', '--break-system-packages'])
        from xgboost import XGBClassifier
        HAS_XGB = True
    except Exception:
        print("  WARNING: XGBoost not available")

HAS_LGBM = False
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    try:
        import subprocess
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lightgbm', '-q'])
        except subprocess.CalledProcessError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lightgbm', '-q', '--break-system-packages'])
        from lightgbm import LGBMClassifier
        HAS_LGBM = True
    except Exception:
        print("  WARNING: LightGBM not available")

print(f"  Models available: LR, GBM, XGB={HAS_XGB}, LGBM={HAS_LGBM}")

# ============================================================
# 1. CONFIGURATION
# ============================================================
if os.path.exists('/kaggle/input'):
    # Try multiple possible Kaggle paths
    for kaggle_path in [
        '/kaggle/input/march-machine-learning-mania-2026',
        '/kaggle/input/competitions/march-machine-learning-mania-2026',
    ]:
        if os.path.exists(kaggle_path):
            BASE = kaggle_path
            break
    else:
        # Auto-detect: find the directory containing MTeams.csv
        import glob
        found = glob.glob('/kaggle/input/*/MTeams.csv')
        if found:
            BASE = os.path.dirname(found[0])
        else:
            BASE = '/kaggle/input/march-machine-learning-mania-2026'
    OUT_DIR = '/kaggle/working'
    OUT_DIR = '/kaggle/working'
else:
    BASE = '/mnt/user-data/uploads'
    OUT_DIR = '/mnt/user-data/outputs'

os.makedirs(OUT_DIR, exist_ok=True)
GOTO_MARGIN = 1.08

stage1_path = f'{BASE}/SampleSubmissionStage1.csv'
stage2_path = f'{BASE}/SampleSubmissionStage2.csv'
if os.path.exists(stage2_path):
    SUBMISSION_PATH = stage2_path
    STAGE = 2
else:
    SUBMISSION_PATH = stage1_path
    STAGE = 1
print(f"  Detected: Stage {STAGE}")

# ============================================================
# 2. DATA LOADING
# ============================================================
print("=" * 60)
print("STEP 1: Loading data...")
print("=" * 60)

m_season_detail = pd.read_csv(f'{BASE}/MRegularSeasonDetailedResults.csv')
m_tourney_compact = pd.read_csv(f'{BASE}/MNCAATourneyCompactResults.csv')
m_compact_season = pd.read_csv(f'{BASE}/MRegularSeasonCompactResults.csv')
m_seeds = pd.read_csv(f'{BASE}/MNCAATourneySeeds.csv')
m_teams = pd.read_csv(f'{BASE}/MTeams.csv')
m_conferences = pd.read_csv(f'{BASE}/MTeamConferences.csv')

w_season_detail = pd.read_csv(f'{BASE}/WRegularSeasonDetailedResults.csv')
w_tourney_compact = pd.read_csv(f'{BASE}/WNCAATourneyCompactResults.csv')
w_compact_season = pd.read_csv(f'{BASE}/WRegularSeasonCompactResults.csv')
w_seeds = pd.read_csv(f'{BASE}/WNCAATourneySeeds.csv')

sub = pd.read_csv(SUBMISSION_PATH)
sub['Season'] = sub['ID'].apply(lambda x: int(x.split('_')[0]))
sub['T1'] = sub['ID'].apply(lambda x: int(x.split('_')[1]))
sub['T2'] = sub['ID'].apply(lambda x: int(x.split('_')[2]))
sub['IsWomens'] = (sub['T1'] >= 3000).astype(int)

print(f"  Submission rows: {len(sub)} (M:{(sub['IsWomens']==0).sum()}, W:{(sub['IsWomens']==1).sum()})")

# ============================================================
# 2b. MASSEY ORDINALS (Men's only)
# ============================================================
print("\n" + "=" * 60)
print("STEP 1b: Loading MasseyOrdinals (Men's only)...")
print("=" * 60)

HAS_MASSEY = False
massey_path = f'{BASE}/MMasseyOrdinals.csv'
if not os.path.exists(massey_path):
    # Fallback search
    for fname in os.listdir(BASE):
        if 'massey' in fname.lower() or 'ordinal' in fname.lower():
            massey_path = f'{BASE}/{fname}'
            break

if os.path.exists(massey_path):
    print(f"  Loading {massey_path}...")
    massey_raw = pd.read_csv(massey_path)
    print(f"  Raw rows: {len(massey_raw):,}")
    print(f"  Systems: {massey_raw['SystemName'].nunique()}")
    print(f"  Seasons: {massey_raw['Season'].min()}-{massey_raw['Season'].max()}")
    HAS_MASSEY = True

    # レギュラーシーズン最終日のランキング (トーナメント前)
    TOURNEY_START_DAY = 134
    massey_reg = massey_raw[massey_raw['RankingDayNum'] < TOURNEY_START_DAY].copy()

    # 各チーム・シーズン・システムの最終ランキング
    latest_day = massey_reg.groupby(['Season', 'SystemName', 'TeamID'])['RankingDayNum'].max().reset_index()
    latest_day.columns = ['Season', 'SystemName', 'TeamID', 'LatestDay']
    massey_latest = massey_reg.merge(latest_day,
                                     left_on=['Season', 'SystemName', 'TeamID', 'RankingDayNum'],
                                     right_on=['Season', 'SystemName', 'TeamID', 'LatestDay'],
                                     how='inner')

    # 主要システム選定
    PRIORITY_SYSTEMS = ['POM', 'SAG', 'WLK', 'COL', 'DOL', 'MOR', 'RPI', 'AP', 'USA',
                        'WOL', 'RTH', 'LMC', 'KPK', 'DCI', 'STH', 'TRK', 'REW', 'SEL']
    available_priority = [s for s in PRIORITY_SYSTEMS
                          if s in massey_latest['SystemName'].unique()]

    # 個別ランクを取得する上位システム
    TOP_N_SYSTEMS = min(5, len(available_priority))
    top_systems = available_priority[:TOP_N_SYSTEMS]

    # 全システム集約
    massey_agg = massey_latest.groupby(['Season', 'TeamID'])['OrdinalRank'].agg(
        MasseyAvgRank='mean',
        MasseyStdRank='std',
        MasseyMinRank='min',
        MasseyMaxRank='max',
        MasseyNumSystems='count',
    ).reset_index()
    massey_agg['MasseyStdRank'] = massey_agg['MasseyStdRank'].fillna(0)
    massey_agg['MasseyRankRange'] = massey_agg['MasseyMaxRank'] - massey_agg['MasseyMinRank']

    # 主要システムpivot
    massey_pivot = massey_latest[massey_latest['SystemName'].isin(top_systems)].pivot_table(
        index=['Season', 'TeamID'],
        columns='SystemName',
        values='OrdinalRank',
        aggfunc='first'
    ).reset_index()
    massey_pivot.columns = ['Season', 'TeamID'] + [f'Rank_{s}' for s in massey_pivot.columns[2:]]

    massey_features = massey_agg.merge(massey_pivot, on=['Season', 'TeamID'], how='left')

    MASSEY_FEAT_COLS = [c for c in massey_features.columns if c not in ['Season', 'TeamID']]

    # 欠損値処理
    for c in MASSEY_FEAT_COLS:
        if c in ['MasseyNumSystems']:
            massey_features[c] = massey_features[c].fillna(0)
        elif c in ['MasseyStdRank', 'MasseyRankRange']:
            massey_features[c] = massey_features[c].fillna(0)
        else:
            massey_features[c] = massey_features[c].fillna(300)

    print(f"  Priority systems found: {available_priority[:10]}...")
    print(f"  Top systems for pivot: {top_systems}")
    print(f"  Massey features: {MASSEY_FEAT_COLS}")
    print(f"  Team-seasons with Massey data: {len(massey_features)}")
else:
    print(f"  WARNING: MasseyOrdinals not found. Men's model will use base features only.")

# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: Computing season stats...")
print("=" * 60)


def compute_season_stats(detail_df):
    w = detail_df.rename(columns={
        'WTeamID': 'TeamID', 'LTeamID': 'OppID',
        'WScore': 'Score', 'LScore': 'OppScore',
        'WFGM': 'FGM', 'WFGA': 'FGA', 'WFGM3': 'FGM3', 'WFGA3': 'FGA3',
        'WFTM': 'FTM', 'WFTA': 'FTA',
        'WOR': 'OR', 'WDR': 'DR', 'WAst': 'Ast', 'WTO': 'TO',
        'WStl': 'Stl', 'WBlk': 'Blk', 'WPF': 'PF',
        'LFGM': 'OppFGM', 'LFGA': 'OppFGA', 'LFGM3': 'OppFGM3', 'LFGA3': 'OppFGA3',
        'LFTM': 'OppFTM', 'LFTA': 'OppFTA',
        'LOR': 'OppOR', 'LDR': 'OppDR', 'LAst': 'OppAst', 'LTO': 'OppTO',
        'LStl': 'OppStl', 'LBlk': 'OppBlk', 'LPF': 'OppPF',
    }).copy()
    w['Win'] = 1
    l = detail_df.rename(columns={
        'LTeamID': 'TeamID', 'WTeamID': 'OppID',
        'LScore': 'Score', 'WScore': 'OppScore',
        'LFGM': 'FGM', 'LFGA': 'FGA', 'LFGM3': 'FGM3', 'LFGA3': 'FGA3',
        'LFTM': 'FTM', 'LFTA': 'FTA',
        'LOR': 'OR', 'LDR': 'DR', 'LAst': 'Ast', 'LTO': 'TO',
        'LStl': 'Stl', 'LBlk': 'Blk', 'LPF': 'PF',
        'WFGM': 'OppFGM', 'WFGA': 'OppFGA', 'WFGM3': 'OppFGM3', 'WFGA3': 'OppFGA3',
        'WFTM': 'OppFTM', 'WFTA': 'OppFTA',
        'WOR': 'OppOR', 'WDR': 'OppDR', 'WAst': 'OppAst', 'WTO': 'OppTO',
        'WStl': 'OppStl', 'WBlk': 'OppBlk', 'WPF': 'OppPF',
    }).copy()
    l['Win'] = 0
    keep = ['Season', 'TeamID', 'Score', 'OppScore', 'Win',
            'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
            'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF',
            'OppFGM', 'OppFGA', 'OppFGM3', 'OppFGA3', 'OppFTM', 'OppFTA',
            'OppOR', 'OppDR', 'OppTO']
    all_g = pd.concat([w[keep], l[keep]], ignore_index=True)
    all_g['Poss'] = all_g['FGA'] - all_g['OR'] + all_g['TO'] + 0.475 * all_g['FTA']
    all_g['OppPoss'] = all_g['OppFGA'] - all_g['OppOR'] + all_g['OppTO'] + 0.475 * all_g['OppFTA']
    agg = all_g.groupby(['Season', 'TeamID']).agg(
        Games=('Win', 'count'), Wins=('Win', 'sum'),
        PtsFor=('Score', 'mean'), PtsAg=('OppScore', 'mean'),
        FGM=('FGM', 'mean'), FGA=('FGA', 'mean'),
        FGM3=('FGM3', 'mean'), FGA3=('FGA3', 'mean'),
        FTM=('FTM', 'mean'), FTA=('FTA', 'mean'),
        OR=('OR', 'mean'), DR=('DR', 'mean'),
        Ast=('Ast', 'mean'), TO=('TO', 'mean'),
        Stl=('Stl', 'mean'), Blk=('Blk', 'mean'), PF=('PF', 'mean'),
        OppFGM=('OppFGM', 'mean'), OppFGA=('OppFGA', 'mean'),
        OppFGM3=('OppFGM3', 'mean'), OppFGA3=('OppFGA3', 'mean'),
        Poss=('Poss', 'mean'), OppPoss=('OppPoss', 'mean'),
        ScoreStd=('Score', 'std'),
    ).reset_index()
    agg['WinPct'] = agg['Wins'] / agg['Games']
    agg['PtDiff'] = agg['PtsFor'] - agg['PtsAg']
    agg['FGPct'] = agg['FGM'] / agg['FGA'].clip(lower=1)
    agg['FG3Pct'] = agg['FGM3'] / agg['FGA3'].clip(lower=1)
    agg['FTPct'] = agg['FTM'] / agg['FTA'].clip(lower=1)
    agg['OppFGPct'] = agg['OppFGM'] / agg['OppFGA'].clip(lower=1)
    agg['OppFG3Pct'] = agg['OppFGM3'] / agg['OppFGA3'].clip(lower=1)
    agg['AstTO'] = agg['Ast'] / agg['TO'].clip(lower=0.1)
    agg['TotReb'] = agg['OR'] + agg['DR']
    agg['OffEff'] = agg['PtsFor'] / agg['Poss'].clip(lower=1) * 100
    agg['DefEff'] = agg['PtsAg'] / agg['Poss'].clip(lower=1) * 100
    agg['NetEff'] = agg['OffEff'] - agg['DefEff']
    agg['FG3Rate'] = agg['FGA3'] / agg['FGA'].clip(lower=1)
    agg['FTRate'] = agg['FTA'] / agg['FGA'].clip(lower=1)
    agg['OppTORate'] = agg['TO'] / agg['Poss'].clip(lower=1)
    agg['StlRate'] = agg['Stl'] / agg['OppPoss'].clip(lower=1)
    agg['ORPct'] = agg['OR'] / (agg['OR'] + agg['DR']).clip(lower=1)
    agg['ScoreStd'] = agg['ScoreStd'].fillna(0)
    return agg


m_stats = compute_season_stats(m_season_detail)
w_stats = compute_season_stats(w_season_detail)
print(f"  Men's team-seasons: {len(m_stats)}")
print(f"  Women's team-seasons: {len(w_stats)}")

# ============================================================
# 3b. OPPONENT-ADJUSTED STATS
# ============================================================
print("\n" + "=" * 60)
print("STEP 2b: Computing opponent-adjusted stats...")
print("=" * 60)


def compute_opp_adjusted(detail_df, stats_df):
    """
    対戦相手の質で補正した統計を計算。
    例: チームAのFG%が45%で、対戦相手の平均被FG%が48%（リーグ平均44%）なら
        AdjFGPct = 45% * (44% / 48%) = 41.25% → 弱い相手のおかげで膨らんでいた分を補正
    """
    # ゲームレベルデータを作成（チーム視点）
    w = detail_df[['Season', 'WTeamID', 'LTeamID', 'WScore', 'LScore',
                    'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA',
                    'WOR', 'WTO',
                    'LFGM', 'LFGA', 'LFGM3', 'LFGA3']].copy()
    w.columns = ['Season', 'TeamID', 'OppID', 'Score', 'OppScore',
                 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
                 'OR_', 'TO_',
                 'OppFGM', 'OppFGA', 'OppFGM3', 'OppFGA3']

    l = detail_df[['Season', 'LTeamID', 'WTeamID', 'LScore', 'WScore',
                    'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA',
                    'LOR', 'LTO',
                    'WFGM', 'WFGA', 'WFGM3', 'WFGA3']].copy()
    l.columns = ['Season', 'TeamID', 'OppID', 'Score', 'OppScore',
                 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
                 'OR_', 'TO_',
                 'OppFGM', 'OppFGA', 'OppFGM3', 'OppFGA3']

    games = pd.concat([w, l], ignore_index=True)

    # ポゼッション
    games['Poss'] = games['FGA'] - games['OR_'] + games['TO_'] + 0.475 * games['FTA']
    games['Poss'] = games['Poss'].clip(lower=1)
    # ゲームレベルの効率
    games['GameOffEff'] = games['Score'] / games['Poss'] * 100
    games['GameDefEff'] = games['OppScore'] / games['Poss'] * 100
    games['GameFGPct'] = games['FGM'] / games['FGA'].clip(lower=1)
    games['GameFG3Pct'] = games['FGM3'] / games['FGA3'].clip(lower=1)
    games['GameOppFGPct'] = games['OppFGM'] / games['OppFGA'].clip(lower=1)

    # 対戦相手のシーズン統計をマージ
    opp_cols = ['Season', 'TeamID', 'OffEff', 'DefEff', 'FGPct', 'FG3Pct', 'OppFGPct']
    opp_stats = stats_df[opp_cols].rename(columns={
        'TeamID': 'OppID',
        'OffEff': 'Opp_SeasonOffEff',
        'DefEff': 'Opp_SeasonDefEff',
        'FGPct': 'Opp_SeasonFGPct',
        'FG3Pct': 'Opp_SeasonFG3Pct',
        'OppFGPct': 'Opp_SeasonOppFGPct',  # 相手チームが許す被FG%
    })
    games = games.merge(opp_stats, on=['Season', 'OppID'], how='left')

    # リーグ平均（シーズンごと）
    league_avg = stats_df.groupby('Season').agg(
        LeagueOffEff=('OffEff', 'mean'),
        LeagueDefEff=('DefEff', 'mean'),
        LeagueFGPct=('FGPct', 'mean'),
        LeagueFG3Pct=('FG3Pct', 'mean'),
        LeagueOppFGPct=('OppFGPct', 'mean'),
    ).reset_index()
    games = games.merge(league_avg, on='Season', how='left')

    # 調整式: AdjStat = GameStat * (LeagueAvg / OpponentSeasonAvg)
    # オフェンス調整: 相手の守備力で補正
    games['AdjGameOffEff'] = games['GameOffEff'] * (
        games['LeagueDefEff'] / games['Opp_SeasonDefEff'].clip(lower=50))
    games['AdjGameFGPct'] = games['GameFGPct'] * (
        games['LeagueOppFGPct'] / games['Opp_SeasonOppFGPct'].clip(lower=0.2))
    games['AdjGameFG3Pct'] = games['GameFG3Pct'] * (
        games['LeagueFG3Pct'] / games['Opp_SeasonFG3Pct'].clip(lower=0.1))

    # ディフェンス調整: 相手の攻撃力で補正
    games['AdjGameDefEff'] = games['GameDefEff'] * (
        games['LeagueOffEff'] / games['Opp_SeasonOffEff'].clip(lower=50))
    games['AdjGameOppFGPct'] = games['GameOppFGPct'] * (
        games['LeagueFGPct'] / games['Opp_SeasonFGPct'].clip(lower=0.2))

    # チーム・シーズンで集約
    adj_agg = games.groupby(['Season', 'TeamID']).agg(
        AdjOffEff=('AdjGameOffEff', 'mean'),
        AdjDefEff=('AdjGameDefEff', 'mean'),
        AdjFGPct=('AdjGameFGPct', 'mean'),
        AdjFG3Pct=('AdjGameFG3Pct', 'mean'),
        AdjOppFGPct=('AdjGameOppFGPct', 'mean'),
    ).reset_index()

    adj_agg['AdjNetEff'] = adj_agg['AdjOffEff'] - adj_agg['AdjDefEff']
    adj_agg['AdjPtDiff'] = (adj_agg['AdjOffEff'] - adj_agg['AdjDefEff']) / 100  # per-possession scale

    return adj_agg


m_adj = compute_opp_adjusted(m_season_detail, m_stats)
w_adj = compute_opp_adjusted(w_season_detail, w_stats)

m_stats = m_stats.merge(m_adj, on=['Season', 'TeamID'], how='left')
w_stats = w_stats.merge(w_adj, on=['Season', 'TeamID'], how='left')

# 欠損埋め
adj_cols = ['AdjOffEff', 'AdjDefEff', 'AdjFGPct', 'AdjFG3Pct', 'AdjOppFGPct', 'AdjNetEff', 'AdjPtDiff']
for c in adj_cols:
    m_stats[c] = m_stats[c].fillna(m_stats[c].median())
    w_stats[c] = w_stats[c].fillna(w_stats[c].median())

print(f"  Men's adjusted stats: {len(m_adj)} team-seasons")
print(f"  Women's adjusted stats: {len(w_adj)} team-seasons")
print(f"  New features: {adj_cols}")

# ============================================================
# 3c. MOMENTUM FEATURES (直近10試合)
# ============================================================
print("\n" + "=" * 60)
print("STEP 2c: Computing momentum features (last 10 games)...")
print("=" * 60)

LAST_N = 10


def compute_momentum(detail_df):
    """直近N試合の統計を計算。シーズン終盤の調子を捕捉。"""
    # 勝ち側
    w = detail_df[['Season', 'DayNum', 'WTeamID', 'WScore', 'LScore',
                    'WFGM', 'WFGA', 'WFGM3', 'WFGA3',
                    'WOR', 'WTO', 'WFTM', 'WFTA']].copy()
    w.columns = ['Season', 'DayNum', 'TeamID', 'Score', 'OppScore',
                 'FGM', 'FGA', 'FGM3', 'FGA3',
                 'OR_', 'TO_', 'FTM', 'FTA']
    w['Win'] = 1

    # 負け側
    l = detail_df[['Season', 'DayNum', 'LTeamID', 'LScore', 'WScore',
                    'LFGM', 'LFGA', 'LFGM3', 'LFGA3',
                    'LOR', 'LTO', 'LFTM', 'LFTA']].copy()
    l.columns = ['Season', 'DayNum', 'TeamID', 'Score', 'OppScore',
                 'FGM', 'FGA', 'FGM3', 'FGA3',
                 'OR_', 'TO_', 'FTM', 'FTA']
    l['Win'] = 0

    games = pd.concat([w, l], ignore_index=True)
    games['PtDiff'] = games['Score'] - games['OppScore']
    games['Poss'] = games['FGA'] - games['OR_'] + games['TO_'] + 0.475 * games['FTA']
    games['Poss'] = games['Poss'].clip(lower=1)
    games['OffEff'] = games['Score'] / games['Poss'] * 100
    games['DefEff'] = games['OppScore'] / games['Poss'] * 100
    games['FGPct'] = games['FGM'] / games['FGA'].clip(lower=1)

    # 各チーム・シーズンの直近N試合を抽出
    games = games.sort_values(['Season', 'TeamID', 'DayNum'])
    last_n = games.groupby(['Season', 'TeamID']).tail(LAST_N)

    mom = last_n.groupby(['Season', 'TeamID']).agg(
        L10_WinPct=('Win', 'mean'),
        L10_PtDiff=('PtDiff', 'mean'),
        L10_OffEff=('OffEff', 'mean'),
        L10_DefEff=('DefEff', 'mean'),
        L10_FGPct=('FGPct', 'mean'),
        L10_Score=('Score', 'mean'),
        L10_Games=('Win', 'count'),
    ).reset_index()

    mom['L10_NetEff'] = mom['L10_OffEff'] - mom['L10_DefEff']

    return mom


m_mom = compute_momentum(m_season_detail)
w_mom = compute_momentum(w_season_detail)

m_stats = m_stats.merge(m_mom, on=['Season', 'TeamID'], how='left')
w_stats = w_stats.merge(w_mom, on=['Season', 'TeamID'], how='left')

mom_cols = ['L10_WinPct', 'L10_PtDiff', 'L10_OffEff', 'L10_DefEff', 'L10_FGPct', 'L10_Score', 'L10_NetEff']
for c in mom_cols:
    m_stats[c] = m_stats[c].fillna(m_stats[c].median())
    w_stats[c] = w_stats[c].fillna(w_stats[c].median())

print(f"  Last {LAST_N} games momentum computed.")
print(f"  Men's: avg L10_WinPct={m_stats['L10_WinPct'].mean():.3f}, L10_PtDiff={m_stats['L10_PtDiff'].mean():.1f}")
print(f"  Women's: avg L10_WinPct={w_stats['L10_WinPct'].mean():.3f}, L10_PtDiff={w_stats['L10_PtDiff'].mean():.1f}")
print(f"  New features: {mom_cols}")

# ============================================================
# 4. ELO RATINGS
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: Computing Elo ratings...")
print("=" * 60)


def compute_elo_by_season(compact_season_df, compact_tourney_df,
                          k_season=20, k_tourney=24, home_adv=100, reversion=0.75):
    elo = {}
    season_end_elo = {}
    reg = compact_season_df.copy(); reg['IsTourney'] = 0
    tny = compact_tourney_df.copy(); tny['IsTourney'] = 1
    all_games = pd.concat([reg, tny], ignore_index=True)
    all_games = all_games.sort_values(['Season', 'DayNum']).reset_index(drop=True)
    current_season = None
    recorded_seasons = set()
    for _, row in all_games.iterrows():
        season = row['Season']
        wid, lid = row['WTeamID'], row['LTeamID']
        is_tourney = row['IsTourney']
        if season != current_season:
            if current_season is not None and current_season not in recorded_seasons:
                for tid, e in elo.items():
                    season_end_elo[(current_season, tid)] = e
                recorded_seasons.add(current_season)
                for tid in elo:
                    elo[tid] = elo[tid] * reversion + 1500 * (1 - reversion)
            current_season = season
        if is_tourney and current_season not in recorded_seasons:
            for tid, e in elo.items():
                season_end_elo[(current_season, tid)] = e
            recorded_seasons.add(current_season)
        for tid in [wid, lid]:
            if tid not in elo:
                elo[tid] = 1500
        k = k_tourney if is_tourney else k_season
        loc_adj = 0
        if 'WLoc' in row.index:
            if row['WLoc'] == 'H': loc_adj = home_adv
            elif row['WLoc'] == 'A': loc_adj = -home_adv
        w_expect = 1.0 / (1.0 + 10.0 ** (-(elo[wid] - elo[lid] + loc_adj) / 400.0))
        elo[wid] += k * (1 - w_expect)
        elo[lid] += k * (0 - (1 - w_expect))
    if current_season not in recorded_seasons:
        for tid, e in elo.items():
            season_end_elo[(current_season, tid)] = e
    return season_end_elo


m_elo = compute_elo_by_season(m_compact_season, m_tourney_compact)
w_elo = compute_elo_by_season(w_compact_season, w_tourney_compact)


def add_elo(stats_df, elo_dict):
    stats_df['Elo'] = stats_df.apply(
        lambda r: elo_dict.get((r['Season'], r['TeamID']), 1500), axis=1)
    return stats_df


m_stats = add_elo(m_stats, m_elo)
w_stats = add_elo(w_stats, w_elo)
print(f"  Men's Elo range: [{m_stats['Elo'].min():.0f}, {m_stats['Elo'].max():.0f}]")
print(f"  Women's Elo range: [{w_stats['Elo'].min():.0f}, {w_stats['Elo'].max():.0f}]")

# ============================================================
# 5. STRENGTH OF SCHEDULE
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: Computing Strength of Schedule...")
print("=" * 60)


def compute_sos(detail_df, elo_dict):
    w = detail_df[['Season', 'WTeamID', 'LTeamID']].copy()
    w.columns = ['Season', 'TeamID', 'OppID']
    l = detail_df[['Season', 'LTeamID', 'WTeamID']].copy()
    l.columns = ['Season', 'TeamID', 'OppID']
    all_g = pd.concat([w, l], ignore_index=True)
    all_g['OppElo'] = all_g.apply(
        lambda r: elo_dict.get((r['Season'], r['OppID']), 1500), axis=1)
    sos = all_g.groupby(['Season', 'TeamID'])['OppElo'].mean().reset_index()
    sos.columns = ['Season', 'TeamID', 'SOS']
    return sos


m_sos = compute_sos(m_season_detail, m_elo)
w_sos = compute_sos(w_season_detail, w_elo)
m_stats = m_stats.merge(m_sos, on=['Season', 'TeamID'], how='left')
w_stats = w_stats.merge(w_sos, on=['Season', 'TeamID'], how='left')
m_stats['SOS'] = m_stats['SOS'].fillna(1500)
w_stats['SOS'] = w_stats['SOS'].fillna(1500)
print(f"  SOS computed.")

# ============================================================
# 5a. CONFERENCE STRENGTH
# ============================================================
print("\n" + "=" * 60)
print("STEP 4a: Computing conference strength...")
print("=" * 60)

m_team_conf = pd.read_csv(f'{BASE}/MTeamConferences.csv')
w_team_conf = pd.read_csv(f'{BASE}/WTeamConferences.csv')


def compute_conf_strength(stats_df, team_conf_df, elo_dict):
    """カンファレンスの平均Eloと所属チーム数を計算"""
    # チームにカンファレンスを紐付け
    tc = team_conf_df[['Season', 'TeamID', 'ConfAbbrev']].copy()

    # Elo付与
    tc['Elo'] = tc.apply(
        lambda r: elo_dict.get((r['Season'], r['TeamID']), 1500), axis=1)

    # カンファレンス×シーズンで集約
    conf_agg = tc.groupby(['Season', 'ConfAbbrev']).agg(
        ConfAvgElo=('Elo', 'mean'),
        ConfMedianElo=('Elo', 'median'),
        ConfMaxElo=('Elo', 'max'),
        ConfTeamCount=('TeamID', 'count'),
    ).reset_index()

    # チームにカンファレンス統計をマージ
    tc_with_conf = tc.merge(conf_agg, on=['Season', 'ConfAbbrev'], how='left')

    # ConfEloRank: カンファレンス内でのチームのElo順位（1=最強）
    tc_with_conf['ConfEloRank'] = tc_with_conf.groupby(
        ['Season', 'ConfAbbrev'])['Elo'].rank(ascending=False)

    # statsにマージ
    conf_feats = tc_with_conf[['Season', 'TeamID',
                                'ConfAvgElo', 'ConfMedianElo', 'ConfMaxElo',
                                'ConfTeamCount', 'ConfEloRank']]

    merged = stats_df.merge(conf_feats, on=['Season', 'TeamID'], how='left')

    # 欠損埋め
    for c in ['ConfAvgElo', 'ConfMedianElo', 'ConfMaxElo']:
        merged[c] = merged[c].fillna(1500)
    merged['ConfTeamCount'] = merged['ConfTeamCount'].fillna(10)
    merged['ConfEloRank'] = merged['ConfEloRank'].fillna(5)

    return merged


m_stats = compute_conf_strength(m_stats, m_team_conf, m_elo)
w_stats = compute_conf_strength(w_stats, w_team_conf, w_elo)

print(f"  Men's top conferences (2025 avg Elo):")
m_tc = m_team_conf[m_team_conf['Season'] == 2025].copy()
m_tc['Elo'] = m_tc.apply(lambda r: m_elo.get((2025, r['TeamID']), 1500), axis=1)
top_conf = m_tc.groupby('ConfAbbrev')['Elo'].mean().sort_values(ascending=False).head(8)
for c, e in top_conf.items():
    print(f"    {c:12s} avg Elo={e:.0f}")
print(f"  New features: ConfAvgElo, ConfMedianElo, ConfMaxElo, ConfTeamCount, ConfEloRank")

# ============================================================
# 5b. MERGE MASSEY INTO MEN'S STATS ONLY
# ============================================================
if HAS_MASSEY:
    print("\n" + "=" * 60)
    print("STEP 4b: Merging Massey features into Men's stats...")
    print("=" * 60)

    m_stats = m_stats.merge(massey_features, on=['Season', 'TeamID'], how='left')
    for c in MASSEY_FEAT_COLS:
        if c in ['MasseyNumSystems']:
            m_stats[c] = m_stats[c].fillna(0)
        elif c in ['MasseyStdRank', 'MasseyRankRange']:
            m_stats[c] = m_stats[c].fillna(0)
        else:
            m_stats[c] = m_stats[c].fillna(300)

    m_has = (m_stats['MasseyNumSystems'] > 0).sum()
    print(f"  Men's team-seasons with Massey data: {m_has}/{len(m_stats)}")

# ============================================================
# 6. SEED PROCESSING
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: Processing seeds...")
print("=" * 60)

m_seeds['SeedNum'] = m_seeds['Seed'].apply(lambda s: int(s[1:3]))
w_seeds['SeedNum'] = w_seeds['Seed'].apply(lambda s: int(s[1:3]))
all_seeds = pd.concat([m_seeds, w_seeds], ignore_index=True)
print(f"  Total seed entries: {len(all_seeds)}")

# Stage 2: 2026 seeds check
if STAGE == 2:
    m_seeds_2026 = m_seeds[m_seeds['Season'] == 2026]
    w_seeds_2026 = w_seeds[w_seeds['Season'] == 2026]
    print(f"  2026 Seeds: Men's={len(m_seeds_2026)}, Women's={len(w_seeds_2026)}")
    if len(m_seeds_2026) == 0:
        print("  WARNING: 2026 seeds not yet available. SeedDiff/SeedHistWinPct will use defaults.")
        print("           Re-run after seeds are released for best predictions.")

# ============================================================
# 6b. SEED MATCHUP HISTORICAL WIN RATES
# ============================================================
print("\n" + "=" * 60)
print("STEP 5b: Computing seed matchup historical win rates...")
print("=" * 60)


def compute_seed_matchup_table(tourney_df, seeds_df):
    """
    各シード対シードの歴史的勝率テーブルを作成。
    返り値: dict[(seed1, seed2)] -> P(seed1 wins)
    seed1 < seed2 (低シード=強い方)
    """
    t = tourney_df.merge(
        seeds_df[['Season', 'TeamID', 'SeedNum']],
        left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID']
    ).rename(columns={'SeedNum': 'WSeed'}).drop('TeamID', axis=1)
    t = t.merge(
        seeds_df[['Season', 'TeamID', 'SeedNum']],
        left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID']
    ).rename(columns={'SeedNum': 'LSeed'}).drop('TeamID', axis=1)

    records = {}  # (lo_seed, hi_seed) -> [wins_for_lo, total]
    for _, r in t.iterrows():
        lo = min(r['WSeed'], r['LSeed'])
        hi = max(r['WSeed'], r['LSeed'])
        key = (lo, hi)
        if key not in records:
            records[key] = [0, 0]
        records[key][1] += 1
        if r['WSeed'] == lo:  # lower seed won
            records[key][0] += 1

    # dict[(lo, hi)] -> P(lower seed wins)
    table = {}
    for (lo, hi), (wins, total) in records.items():
        table[(lo, hi)] = wins / total
    return table


def get_seed_hist_winpct(seed1, seed2, table):
    """
    seed1 vs seed2 の試合で seed1 が勝つ歴史的確率を返す。
    テーブルにないペアは 0.5 を返す。
    """
    if seed1 == seed2:
        return 0.5
    lo, hi = min(seed1, seed2), max(seed1, seed2)
    lo_wins = table.get((lo, hi), 0.5)
    if seed1 == lo:
        return lo_wins  # seed1が強い方
    else:
        return 1.0 - lo_wins  # seed1が弱い方


m_seed_table = compute_seed_matchup_table(m_tourney_compact, m_seeds)
w_seed_table = compute_seed_matchup_table(w_tourney_compact, w_seeds)

print("  Men's key matchups:")
for s1, s2 in [(1,16),(2,15),(3,14),(4,13),(5,12),(6,11),(7,10),(8,9)]:
    wp = m_seed_table.get((s1, s2), 0.5)
    print(f"    {s1:>2d} vs {s2:>2d}: {wp:.1%}")
print("  Women's key matchups:")
for s1, s2 in [(1,16),(2,15),(3,14),(4,13),(5,12),(6,11),(7,10),(8,9)]:
    wp = w_seed_table.get((s1, s2), 0.5)
    print(f"    {s1:>2d} vs {s2:>2d}: {wp:.1%}")

# ============================================================
# 7. BUILD TRAINING DATA
# ============================================================
print("\n" + "=" * 60)
print("STEP 6: Building training data (separate M/W)...")
print("=" * 60)

# Women's: 30 base + 7 opponent-adjusted + 7 momentum
W_FEAT_COLS = [
    'WinPct', 'PtDiff', 'PtsFor', 'PtsAg',
    'FGPct', 'FG3Pct', 'FTPct', 'OppFGPct', 'OppFG3Pct',
    'AstTO', 'OR', 'DR', 'TotReb', 'Ast', 'TO', 'Stl', 'Blk',
    'OffEff', 'DefEff', 'NetEff', 'Poss',
    'FG3Rate', 'FTRate', 'OppTORate', 'StlRate', 'ORPct',
    'ScoreStd', 'Elo', 'SOS',
    # Opponent-adjusted stats
    'AdjOffEff', 'AdjDefEff', 'AdjNetEff',
    'AdjFGPct', 'AdjFG3Pct', 'AdjOppFGPct', 'AdjPtDiff',
    # Momentum (last 10 games)
    'L10_WinPct', 'L10_PtDiff', 'L10_OffEff', 'L10_DefEff',
    'L10_FGPct', 'L10_Score', 'L10_NetEff',
    # Conference strength
    'ConfAvgElo', 'ConfMedianElo', 'ConfMaxElo',
    'ConfTeamCount', 'ConfEloRank',
]
w_feat_names = ['SeedDiff', 'SeedHistWinPct'] + [f'{c}_d' for c in W_FEAT_COLS]

# Men's: v1の30特徴量 + Massey特徴量
M_FEAT_COLS = W_FEAT_COLS.copy()
if HAS_MASSEY:
    M_FEAT_COLS = M_FEAT_COLS + MASSEY_FEAT_COLS
m_feat_names = ['SeedDiff', 'SeedHistWinPct'] + [f'{c}_d' for c in M_FEAT_COLS]


def build_training(tourney_df, stats_df, seeds_df, feat_cols, seed_table, min_season=2003):
    rows = []
    avail = set(stats_df['Season'].unique())
    for _, g in tourney_df.iterrows():
        s = g['Season']
        if s < min_season or s not in avail:
            continue
        wid, lid = g['WTeamID'], g['LTeamID']
        wr = stats_df[(stats_df['Season'] == s) & (stats_df['TeamID'] == wid)]
        lr_ = stats_df[(stats_df['Season'] == s) & (stats_df['TeamID'] == lid)]
        if len(wr) == 0 or len(lr_) == 0:
            continue
        wr, lr_ = wr.iloc[0], lr_.iloc[0]
        ws = seeds_df[(seeds_df['Season'] == s) & (seeds_df['TeamID'] == wid)]
        ls = seeds_df[(seeds_df['Season'] == s) & (seeds_df['TeamID'] == lid)]
        w_seed = ws['SeedNum'].values[0] if len(ws) > 0 else 8.5
        l_seed = ls['SeedNum'].values[0] if len(ls) > 0 else 8.5

        # Sample 1: Team1=winner, Label=1
        f1 = {'Season': s, 'SeedDiff': w_seed - l_seed,
              'SeedHistWinPct': get_seed_hist_winpct(w_seed, l_seed, seed_table)}
        for c in feat_cols:
            f1[f'{c}_d'] = wr[c] - lr_[c]
        f1['Label'] = 1
        rows.append(f1)

        # Sample 2: Team1=loser, Label=0
        f2 = {'Season': s, 'SeedDiff': l_seed - w_seed,
              'SeedHistWinPct': get_seed_hist_winpct(l_seed, w_seed, seed_table)}
        for c in feat_cols:
            f2[f'{c}_d'] = lr_[c] - wr[c]
        f2['Label'] = 0
        rows.append(f2)
    return pd.DataFrame(rows)


m_train = build_training(m_tourney_compact, m_stats, m_seeds, M_FEAT_COLS, m_seed_table)
w_train = build_training(w_tourney_compact, w_stats, w_seeds, W_FEAT_COLS, w_seed_table)

print(f"  Men's training:   {len(m_train)} samples, {len(m_feat_names)} features")
print(f"  Women's training: {len(w_train)} samples, {len(w_feat_names)} features")
if HAS_MASSEY:
    print(f"  Men's extra Massey features: {len(MASSEY_FEAT_COLS)}")

# ============================================================
# 8. MODEL TRAINING (男女別)
# ============================================================
print("\n" + "=" * 60)
print("STEP 7: Training separate M/W models...")
print("=" * 60)


def train_models(train_df, feat_names_list, label, decay=0.97):
    X = train_df[feat_names_list].values.astype(np.float64)
    y = train_df['Label'].values.astype(np.float64)
    X = np.nan_to_num(X, nan=0.0)

    # 直近シーズンほど重みを大きく
    seasons = train_df['Season'].values
    max_season = seasons.max()
    sample_weight = np.array([decay ** (max_season - s) for s in seasons])
    print(f"\n  [{label}] Season weighting (decay={decay}):")
    print(f"    {int(max_season)}: weight=1.000, {int(max_season)-10}: weight={decay**10:.3f}, {int(max_season)-20}: weight={decay**20:.3f}")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Model 1: Logistic Regression
    lr = LogisticRegression(C=0.5, max_iter=2000, random_state=42)
    lr.fit(X_scaled, y, sample_weight=sample_weight)

    # Model 2: sklearn GBM
    gb = GradientBoostingClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.04,
        subsample=0.8, min_samples_leaf=15,
        max_features='sqrt', random_state=42
    )
    gb.fit(X, y, sample_weight=sample_weight)

    # Model 3: XGBoost
    xgb = None
    if HAS_XGB:
        xgb = XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.04,
            subsample=0.8, colsample_bytree=0.7,
            min_child_weight=15, reg_alpha=0.1, reg_lambda=1.0,
            random_state=42, verbosity=0, eval_metric='logloss'
        )
        xgb.fit(X, y, sample_weight=sample_weight)

    # Model 4: LightGBM
    lgbm = None
    if HAS_LGBM:
        lgbm = LGBMClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.04,
            subsample=0.8, colsample_bytree=0.7,
            min_child_samples=15, reg_alpha=0.1, reg_lambda=1.0,
            random_state=42, verbosity=-1
        )
        lgbm.fit(X, y, sample_weight=sample_weight)

    imp = sorted(zip(feat_names_list, gb.feature_importances_),
                 key=lambda x: -x[1])[:10]
    print(f"\n  [{label}] Top 10 features (GBM):")
    for fname, fval in imp:
        print(f"    {fname:25s} {fval:.4f}")

    n_models = 2 + (1 if xgb else 0) + (1 if lgbm else 0)
    print(f"  [{label}] Ensemble: {n_models} models (LR, GBM{', XGB' if xgb else ''}{', LGBM' if lgbm else ''})")

    return lr, gb, xgb, lgbm, scaler


print("  --- Men's Model (30 base + Massey) ---")
m_lr, m_gb, m_xgb, m_lgbm, m_scaler = train_models(m_train, m_feat_names, "MEN")

print("\n  --- Women's Model (30 base + SeedHistWinPct) ---")
w_lr, w_gb, w_xgb, w_lgbm, w_scaler = train_models(w_train, w_feat_names, "WOMEN")

# ============================================================
# 9. goto_conversion
# ============================================================
print("\n" + "=" * 60)
print("STEP 8: goto_conversion calibration...")
print("=" * 60)


def preprocessed_goto_conversion(list_of_odds, total=1):
    probs = [1.0 / x for x in list_of_odds]
    if sum(probs) < total:
        reversed_odds = [1.0 / (1.0 - x) for x in probs]
        reversed_probs = goto_conversion.goto_conversion(
            reversed_odds, multiplicativeIfImprudentOdds=True)
        return [1.0 - x for x in reversed_probs]
    else:
        return goto_conversion.goto_conversion(
            list_of_odds, multiplicativeIfImprudentOdds=True)


def apply_goto_calibration(p, margin=GOTO_MARGIN):
    if p < 0.01 or p > 0.99:
        return p
    p1 = p * margin
    p2 = (1 - p) * margin
    if p1 >= 1.0 or p2 >= 1.0:
        return p
    odds = [1.0 / p1, 1.0 / p2]
    try:
        calibrated = preprocessed_goto_conversion(odds)
        return calibrated[0]
    except Exception:
        return p


print(f"  goto_conversion margin: {GOTO_MARGIN}")

# ============================================================
# 10. GENERATE PREDICTIONS
# ============================================================
print("\n" + "=" * 60)
print("STEP 9: Generating predictions...")
print("=" * 60)

m_stats_idx = {}
for _, r in m_stats.iterrows():
    m_stats_idx[(r['Season'], r['TeamID'])] = r
w_stats_idx = {}
for _, r in w_stats.iterrows():
    w_stats_idx[(r['Season'], r['TeamID'])] = r
seeds_idx = {}
for _, r in all_seeds.iterrows():
    seeds_idx[(r['Season'], r['TeamID'])] = r['SeedNum']

m_mask = (sub['IsWomens'] == 0).values
w_mask = (sub['IsWomens'] == 1).values

# 男女別に特徴量行列を構築
# Men's
m_feat_matrix = np.zeros((len(sub), len(m_feat_names)), dtype=np.float64)
# Women's
w_feat_matrix = np.zeros((len(sub), len(w_feat_names)), dtype=np.float64)
has_data = np.ones(len(sub), dtype=bool)

for i, row in enumerate(sub.itertuples()):
    s, t1, t2 = row.Season, row.T1, row.T2
    is_w = row.IsWomens

    stats_idx_cur = w_stats_idx if is_w else m_stats_idx
    r1 = stats_idx_cur.get((s, t1))
    r2 = stats_idx_cur.get((s, t2))

    if r1 is None or r2 is None:
        has_data[i] = False
        continue

    sd1 = seeds_idx.get((s, t1), 8.5)
    sd2 = seeds_idx.get((s, t2), 8.5)

    if is_w:
        w_feat_matrix[i, 0] = sd1 - sd2
        w_feat_matrix[i, 1] = get_seed_hist_winpct(sd1, sd2, w_seed_table)
        for j, c in enumerate(W_FEAT_COLS):
            w_feat_matrix[i, j + 2] = r1[c] - r2[c]
    else:
        m_feat_matrix[i, 0] = sd1 - sd2
        m_feat_matrix[i, 1] = get_seed_hist_winpct(sd1, sd2, m_seed_table)
        for j, c in enumerate(M_FEAT_COLS):
            m_feat_matrix[i, j + 2] = r1[c] - r2[c]

m_feat_matrix = np.nan_to_num(m_feat_matrix, nan=0.0)
w_feat_matrix = np.nan_to_num(w_feat_matrix, nan=0.0)

raw_pred = np.full(len(sub), 0.5)


def ensemble_predict(lr, gb, xgb, lgbm, scaler, feat, feat_raw):
    """4モデルアンサンブル。利用可能なモデルで等重みブレンド。"""
    preds = []
    weights = []

    # LR: weight 0.20
    preds.append(lr.predict_proba(scaler.transform(feat_raw))[:, 1])
    weights.append(0.20)

    # GBM: weight 0.30
    preds.append(gb.predict_proba(feat_raw)[:, 1])
    weights.append(0.30)

    # XGB: weight 0.25
    if xgb is not None:
        preds.append(xgb.predict_proba(feat_raw)[:, 1])
        weights.append(0.25)

    # LGBM: weight 0.25
    if lgbm is not None:
        preds.append(lgbm.predict_proba(feat_raw)[:, 1])
        weights.append(0.25)

    # 正規化
    w_sum = sum(weights)
    weights = [w / w_sum for w in weights]

    result = np.zeros(len(preds[0]))
    for p, w in zip(preds, weights):
        result += w * p
    return result


# Men's predictions
m_idx = np.where(m_mask & has_data)[0]
if len(m_idx) > 0:
    m_feat = m_feat_matrix[m_idx]
    raw_pred[m_idx] = ensemble_predict(m_lr, m_gb, m_xgb, m_lgbm, m_scaler, m_feat, m_feat)

# Women's predictions
w_idx = np.where(w_mask & has_data)[0]
if len(w_idx) > 0:
    w_feat = w_feat_matrix[w_idx]
    raw_pred[w_idx] = ensemble_predict(w_lr, w_gb, w_xgb, w_lgbm, w_scaler, w_feat, w_feat)

print(f"  Men's raw pred:   mean={raw_pred[m_mask].mean():.4f}  std={raw_pred[m_mask].std():.4f}")
print(f"  Women's raw pred: mean={raw_pred[w_mask].mean():.4f}  std={raw_pred[w_mask].std():.4f}")

# goto_conversion
print("  Applying goto_conversion...")
calibrated_pred = np.array([apply_goto_calibration(p) for p in raw_pred])
calibrated_pred = np.clip(calibrated_pred, 0.025, 0.975)
calibrated_pred[~has_data] = 0.5

print(f"\n  Raw predictions:        mean={raw_pred.mean():.4f}  std={raw_pred.std():.4f}")
print(f"  Calibrated predictions: mean={calibrated_pred.mean():.4f}  std={calibrated_pred.std():.4f}")
print(f"  Men's calibrated:   mean={calibrated_pred[m_mask].mean():.4f}  std={calibrated_pred[m_mask].std():.4f}")
print(f"  Women's calibrated: mean={calibrated_pred[w_mask].mean():.4f}  std={calibrated_pred[w_mask].std():.4f}")

# ============================================================
# 11. SAVE
# ============================================================
print("\n" + "=" * 60)
print("STEP 10: Saving submission...")
print("=" * 60)

sub['Pred'] = calibrated_pred
output_path = f'{OUT_DIR}/submission.csv'
sub[['ID', 'Pred']].to_csv(output_path, index=False)

print(f"  Saved to: {output_path}")
print(f"  Predictions: {len(sub)}")
print(f"  Mean:  {sub['Pred'].mean():.5f}")
print(f"  Std:   {sub['Pred'].std():.5f}")
print(f"  Min:   {sub['Pred'].min():.5f}")
print(f"  Max:   {sub['Pred'].max():.5f}")
print(f"  Missing: {(~has_data).sum()}")

n_massey = len(m_feat_names) - 32 if HAS_MASSEY else 0
print(f"\n  Men's: {len(m_feat_names)} features (30 base + SeedHistWinPct + {n_massey} Massey)")
print(f"  Women's: {len(w_feat_names)} features (30 base + SeedHistWinPct)")

print("\n  Sample predictions:")
sample = sub[has_data].copy()
sample_extreme = pd.concat([
    sample.nlargest(5, 'Pred')[['ID', 'Pred']],
    sample.nsmallest(5, 'Pred')[['ID', 'Pred']],
])
print(sample_extreme.to_string(index=False))

print("\n" + "=" * 60)
print("DONE!")
print("=" * 60)

  Models available: LR, GBM, XGB=True, LGBM=True
  Detected: Stage 2
STEP 1: Loading data...
  Submission rows: 132133 (M:66430, W:65703)

STEP 1b: Loading MasseyOrdinals (Men's only)...
  Loading /kaggle/input/competitions/march-machine-learning-mania-2026/MMasseyOrdinals.csv...
  Raw rows: 5,819,228
  Systems: 196
  Seasons: 2003-2026
  Priority systems found: ['POM', 'SAG', 'WLK', 'COL', 'DOL', 'MOR', 'RPI', 'AP', 'USA', 'WOL']...
  Top systems for pivot: ['POM', 'SAG', 'WLK', 'COL', 'DOL']
  Massey features: ['MasseyAvgRank', 'MasseyStdRank', 'MasseyMinRank', 'MasseyMaxRank', 'MasseyNumSystems', 'MasseyRankRange', 'Rank_COL', 'Rank_DOL', 'Rank_POM', 'Rank_SAG', 'Rank_WLK']
  Team-seasons with Massey data: 8356

STEP 2: Computing season stats...
  Men's team-seasons: 8346
  Women's team-seasons: 5965

STEP 2b: Computing opponent-adjusted stats...
  Men's adjusted stats: 8346 team-seasons
  Women's adjusted stats: 5965 team-seasons
  New features: ['AdjOffEff', 'AdjDefEff', 'AdjFGPct